# Optimized Qwen3 merger — Kaggle smoke test
Enable Internet and GPU T4 x2. Default: one epoch, at most 30 optimizer steps and a 30-minute training budget. Search and evaluation add runtime. This pilot cannot statistically establish the five-point accuracy target. Re-run cells to resume verified artifacts.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
ROOT=Path('/kaggle/working')
REPO=ROOT/'Layer-Merging'
OUTPUT=ROOT/'outputs/smoke'
print('Python:',sys.version)
subprocess.run(['nvidia-smi'],check=True)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/amritosanto/Layer-Merging.git',str(REPO)],check=True)
else:
    if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
        raise RuntimeError('Repository has local edits; preserve them before updating')
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source revision:',REVISION)


## Install binary wheels without replacing CUDA PyTorch

In [ ]:
PACKAGES = ['transformers==4.51.3', 'tokenizers==0.21.4', 'datasets==4.4.1', 'accelerate==1.6.0', 'bitsandbytes==0.48.2', 'huggingface-hub==0.36.0', 'numpy==2.2.6', 'pandas==2.2.3', 'matplotlib==3.10.1', 'safetensors==0.5.3', 'pytest==8.3.5', 'pyarrow==21.0.0', 'python-dateutil==2.9.0.post0', 'dill==0.4.0', 'multiprocess==0.70.18', 'fsspec==2025.9.0']
torch_version=importlib.metadata.version('torch')
constraint=ROOT/'qwen-keep-torch.txt'
constraint.write_text('torch=='+torch_version+'\n')
subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:','-c',str(constraint),*PACKAGES],check=True)
assert importlib.metadata.version('torch')==torch_version
ENV=os.environ.copy()
ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1')


## Configuration and GPU check
Edit the small overrides below before starting a new output directory. Wiki-40B configurations are discovered before selecting en/hi/ja where available. Benchmark subsets are intersected with published configurations.

In [ ]:
config=json.loads((REPO/'config.smoke.json').read_text())
config['output_dir']=str(OUTPUT)
# Optional overrides for a NEW experiment:
# config['max_optimizer_steps']=30
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG=ROOT/'qwen-smoke-run.json'
CONFIG.write_text(json.dumps(config,indent=2))
print(json.dumps(config,indent=2))
def run_stage(stage):
    subprocess.run([sys.executable,'run.py',stage,'--config',str(CONFIG)],cwd=REPO,env=ENV,check=True)
subprocess.run([sys.executable,'-c','from config import Config; from src.trainer import hardware; import sys; hardware(Config.load(sys.argv[1]))',str(CONFIG)],cwd=REPO,env=ENV,check=True)
# One-command alternative to the following stage cells: run_stage('smoke')


## Train briefly — or reuse completed training

In [ ]:
run_stage('train')


## CE search and one final NF4 export — cached on resume

In [ ]:
run_stage('rank-export')


## Inspect CE candidates and selected plan

In [ ]:
import csv
from IPython.display import display, Markdown, FileLink
for name in ['single_pair_ce.csv','candidate_plans.csv']:
    p=OUTPUT/'rankings'/name
    with p.open() as stream: display(list(csv.DictReader(stream)))
selected=json.loads((OUTPUT/'rankings/selected_plan.json').read_text())
print(json.dumps(selected,indent=2))


## Benchmark only final learned NF4 and original FP16 reference

In [ ]:
run_stage('benchmark')
run_stage('report')


## Results and saved artifacts
Download the output folder through Kaggle’s Output panel to retain model shards and checkpoints. No additional copies or archives of the model are created.

In [ ]:
display(Markdown((OUTPUT/'REPORT.md').read_text()))
for name in ['accuracy_comparison.csv','per_language_accuracy.csv','answer_option_frequencies.csv']:
    path=OUTPUT/'benchmark_results'/config['evaluation_mode']/name
    with path.open() as stream: display(list(csv.DictReader(stream)))
for path in [OUTPUT/'REPORT.md',OUTPUT/'checkpoints/merger_final.pt',OUTPUT/'rankings/selected_plan.json']:
    display(FileLink(str(path)))
for path in sorted((OUTPUT/'model_exports/best_learned_nf4').iterdir()):
    if path.is_file():display(FileLink(str(path)))
